# Submission 08 - Experiment 23 Family/Ticket Aware Ensemble

This submission uses the final full-data model from Experiment 23. It combines XGBoost and CatBoost using equal-weight probability averaging. The feature pipeline uses family, surname, ticket, and family-ticket target-derived features. No external Titanic labels are used.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

SEED = 42

ROOT = Path.cwd().parent
DATA = ROOT / 'data'
SUBMISSIONS = ROOT / 'submissions'
SUBMISSIONS.mkdir(exist_ok=True)

train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')

y = train['Survived'].astype(int)
base = train.drop(columns=['Survived']).copy()
test_base = test.copy()


In [ ]:
def engineer(df):
    x = df.copy()

    x['FamilySize'] = x['SibSp'] + x['Parch'] + 1
    x['IsAlone'] = (x['FamilySize'] == 1).astype(int)
    x['Child'] = (x['Age'] < 14).astype(int)

    x['Title'] = x['Name'].str.extract(r',\s*([^.]*)\.')[0].str.strip()
    x['Title'] = x['Title'].replace({
        'Mlle': 'Miss',
        'Ms': 'Miss',
        'Mme': 'Mrs'
    })

    common_titles = ['Mr', 'Miss', 'Mrs', 'Master']
    x['Title'] = x['Title'].where(
        x['Title'].isin(common_titles),
        'Rare'
    )

    x['Surname'] = x['Name'].str.split(',').str[0].str.strip()

    x['TicketPrefix'] = (
        x['Ticket'].astype(str)
        .str.replace(r'\d', '', regex=True)
        .str.replace(r'[./]+', '', regex=True)
        .str.replace(r'\s+', '', regex=True)
        .str.upper()
    )
    x['TicketPrefix'] = x['TicketPrefix'].replace('', 'NONE')

    x['TicketGroupSize'] = x.groupby('Ticket')['PassengerId'].transform('count')
    x['SurnameGroupSize'] = x.groupby('Surname')['PassengerId'].transform('count')
    x['FamilyTicket'] = x['Surname'].astype(str) + '_' + x['Ticket'].astype(str)

    x['FarePerPerson'] = x['Fare'] / x['TicketGroupSize'].replace(0, np.nan)
    x['SexPclass'] = x['Sex'].astype(str) + '_' + x['Pclass'].astype(str)
    x['FamilySex'] = x['FamilySize'].astype(str) + '_' + x['Sex'].astype(str)
    x['PclassTitle'] = x['Pclass'].astype(str) + '_' + x['Title'].astype(str)
    x['AgeBand'] = pd.cut(
        x['Age'],
        bins=[-1, 5, 12, 18, 30, 50, 100],
        labels=False
    )
    x['FareBand'] = pd.qcut(
        x['Fare'],
        q=5,
        labels=False,
        duplicates='drop'
    )

    x['NameLength'] = x['Name'].astype(str).str.len()
    x['NameWords'] = x['Name'].astype(str).str.split().str.len()
    x['TicketLength'] = x['Ticket'].astype(str).str.len()
    x['DeckKnown'] = x['Cabin'].notna().astype(int)
    x['CabinDeck'] = x['Cabin'].astype(str).str[0].replace('n', 'Unknown')
    x['LargeFamily'] = (x['FamilySize'] >= 5).astype(int)
    x['SmallFamily'] = x['FamilySize'].between(2, 4).astype(int)
    x['FemaleChild'] = ((x['Sex'] == 'female') & (x['Age'] < 18)).astype(int)
    x['FarePerAge'] = x['Fare'] / x['Age'].replace(0, np.nan)
    x['ClassFare'] = x['Pclass'] * x['Fare']
    x['SiblingChildRatio'] = x['SibSp'] / (x['Parch'] + 1)
    x['FamilyFare'] = x['Fare'] * x['FamilySize']
    x['SexTitle'] = x['Sex'].astype(str) + '_' + x['Title'].astype(str)

    x['Mother'] = (
        (x['Sex'] == 'female') &
        (x['Age'] > 18) &
        (x['Parch'] > 0) &
        (x['Title'] != 'Miss')
    ).astype(int)

    x['AgeMissing'] = x['Age'].isna().astype(int)
    x['FareMissing'] = x['Fare'].isna().astype(int)
    x['EmbarkedMissing'] = x['Embarked'].isna().astype(int)
    x['HasCabin'] = x['Cabin'].notna().astype(int)

    return x


In [ ]:
def group_maps(ref, yref, smoothing=5):
    gm = float(yref.mean())
    maps = {}

    for col in ['Ticket', 'Surname', 'FamilyTicket']:
        tmp = pd.DataFrame({
            col: ref[col].astype(str).values,
            'y': yref.values
        })

        stats = tmp.groupby(col)['y'].agg(['mean', 'count'])
        stats['smooth'] = (
            (stats['mean'] * stats['count'] + gm * smoothing) /
            (stats['count'] + smoothing)
        )

        maps[col] = stats['smooth'].to_dict()

    return maps, gm


def apply_maps(df, maps, gm):
    x = df.copy()

    for col in ['Ticket', 'Surname', 'FamilyTicket']:
        x[col + 'SurvivalTE'] = (
            x[col].astype(str).map(maps[col]).fillna(gm)
        )

    return x


def safe_features(tr, ytr, va):
    tr2 = tr.copy()
    va2 = va.copy()

    inner = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=SEED
    )

    oof = pd.DataFrame(index=tr2.index)

    for col in ['Ticket', 'Surname', 'FamilyTicket']:
        oof[col + 'SurvivalTE'] = np.nan

    for itr, iva in inner.split(tr2, ytr):
        ref = tr2.iloc[itr]
        yref = ytr.iloc[itr]
        maps, gm = group_maps(ref, yref)
        mapped = apply_maps(tr2.iloc[iva], maps, gm)

        for col in ['Ticket', 'Surname', 'FamilyTicket']:
            oof.loc[
                tr2.index[iva],
                col + 'SurvivalTE'
            ] = mapped[col + 'SurvivalTE'].values

    maps, gm = group_maps(tr2, ytr)
    val_maps = apply_maps(va2, maps, gm)

    for col in ['Ticket', 'Surname', 'FamilyTicket']:
        counts = tr2[col].astype(str).value_counts()

        oof[col + 'Count'] = (
            tr2[col].astype(str).map(counts).fillna(0).values
        )

        val_maps[col + 'Count'] = (
            va2[col].astype(str).map(counts).fillna(0).values
        )

    tr_out = pd.concat(
        [tr2.reset_index(drop=True), oof.reset_index(drop=True)],
        axis=1
    )

    va_out = pd.concat(
        [va2.reset_index(drop=True), val_maps.reset_index(drop=True)],
        axis=1
    )

    return tr_out, va_out


In [ ]:
base = engineer(base)
test_base = engineer(test_base)

train_features, test_features = safe_features(
    base,
    y,
    test_base
)

DROP = [
    'PassengerId',
    'Name',
    'Ticket',
    'Cabin',
    'Surname',
    'FamilyTicket'
]

def prep(df):
    x = df.drop(
        columns=[c for c in DROP if c in df.columns]
    ).copy()

    for c in x.select_dtypes('object').columns:
        x[c] = x[c].fillna('Unknown').astype(str)

    return x

X_train = prep(train_features)
X_test = prep(test_features)
X_test = X_test.reindex(columns=X_train.columns)


In [ ]:
# XGBoost
cat_cols = X_train.select_dtypes('object').columns.tolist()
num_cols = [c for c in X_train.columns if c not in cat_cols]

pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='most_frequent')),
        ('oh', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])

xgb = XGBClassifier(
    n_estimators=900,
    max_depth=3,
    learning_rate=0.025,
    subsample=0.82,
    colsample_bytree=0.85,
    min_child_weight=3,
    gamma=0.05,
    reg_alpha=0.05,
    reg_lambda=2.5,
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=SEED,
    n_jobs=-1
)

xp = Pipeline([
    ('pre', pre),
    ('model', xgb)
])

xp.fit(X_train, y)
p_xgb = xp.predict_proba(X_test)[:, 1]


In [ ]:
# CatBoost
A = X_train.copy()
B = X_test.copy()

cb_cols = A.select_dtypes('object').columns.tolist()

for c in cb_cols:
    A[c] = A[c].fillna('Unknown').astype(str)
    B[c] = B[c].fillna('Unknown').astype(str)

for c in A.columns:
    if c not in cb_cols:
        A[c] = A[c].replace([np.inf, -np.inf], np.nan)
        B[c] = B[c].replace([np.inf, -np.inf], np.nan)
        med = A[c].median()
        A[c] = A[c].fillna(med)
        B[c] = B[c].fillna(med)

cb = CatBoostClassifier(
    iterations=900,
    depth=5,
    learning_rate=0.025,
    l2_leaf_reg=7,
    loss_function='Logloss',
    verbose=False,
    random_seed=SEED,
    thread_count=-1
)

cb.fit(
    A,
    y,
    cat_features=cb_cols
)

p_cat = cb.predict_proba(B)[:, 1]

# Experiment 23 final equal-weight blend
p_blend = 0.5 * p_xgb + 0.5 * p_cat
pred = (p_blend >= 0.5).astype(int)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': pred
})

assert len(submission) == 418
assert list(submission.columns) == ['PassengerId', 'Survived']
assert submission['PassengerId'].equals(test['PassengerId'])
assert set(submission['Survived'].unique()).issubset({0, 1})

path = SUBMISSIONS / 'submission_08.csv'
submission.to_csv(path, index=False)

print(f'Saved: {path}')
print(f'Test rows: {len(submission)}')
print(f'Predicted survived: {(pred == 1).sum()}')
print(f'Predicted not survived: {(pred == 0).sum()}')
